In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import torch
from utils import *
from experiment import Experiment
from reasoning_graph import ReasoningGraph
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# model_name = "Qwen/Qwen3-4B-Thinking-2507"
# model_name = "Qwen/Qwen3-1.7B"
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
# dataset_name = 'yentinglin/aime_2025'
# dataset_name = 'HuggingFaceH4/aime_2024'
# dataset_name = "HuggingFaceH4/MATH-500"
dataset_name = "qwedsacf/competition_math"
problems = "5"
num_rollouts = 10
temperature = 0.6

In [4]:
problems = load_problems_from_dataset(dataset_name=dataset_name, problems=problems, level="Level 3")
problems



Loading 5 problems from qwedsacf/competition_math:test
Applied filters {'level': 'Level 3'}: 12500 -> 2723 problems


[{'index': 1580,
  'question': 'Find\n\\[\\sin \\left( \\sin^{-1} \\frac{3}{5} + \\tan^{-1} 2 \\right).\\]',
  'full_answer': 'Let $a = \\sin^{-1} \\frac{3}{5}$ and $b = \\tan^{-1} 2.$  Then $\\sin a = \\frac{3}{5}$ and $\\tan b = 2.$  With the usual technique of constructing right triangles, we can find that $\\cos a = \\frac{4}{5},$ $\\cos b = \\frac{1}{\\sqrt{5}},$ and $\\sin b = \\frac{2}{\\sqrt{5}}.$  Therefore, from the angle addition formula,\n\\begin{align*}\n\\sin (a + b) &= \\sin a \\cos b + \\cos a \\sin b \\\\\n&= \\frac{3}{5} \\cdot \\frac{1}{\\sqrt{5}} + \\frac{4}{5} \\cdot \\frac{2}{\\sqrt{5}} \\\\\n&= \\frac{11}{5 \\sqrt{5}} \\\\\n&= \\boxed{\\frac{11 \\sqrt{5}}{25}}.\n\\end{align*}',
  'ground_truth': '\\frac{11 \\sqrt{5}}{25}',
  'level': 'Level 3'},
 {'index': 33,
  'question': 'You have a whole pizza in the refrigerator. On your first trip to the refrigerator, you eat half the pizza. On each successive trip, you eat half of the remaining pizza. After five trips to t

In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [6]:
# Create an experiment
experiment = Experiment()

# Setup new experiment
experiment.setup_new(
    model_name=model_name,
    dataset_name=dataset_name,
    problems=problems,
    num_rollouts=num_rollouts,
    temperature=temperature,
    max_new_tokens=8192,
    condition_on_final_answer=False
)

Setup new experiment at: experiments/experiment_20251113_002807 with config:
{'model_name': 'Qwen/Qwen2.5-Math-1.5B-Instruct', 'dataset_name': 'qwedsacf/competition_math', 'problems': None, 'num_problems': 5, 'num_rollouts': 10, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'min_p': 0, 'max_new_tokens': 8192, 'store_probs_logits': False, 'condition_on_final_answer': False, 'timestamp': '20251113_002807'}


In [ ]:
# Conduct the experiment with initiailized model and tokenizer
experiment.conduct_experiment(model, tokenizer)


Problem 1/5


Problem 1580:   0%|          | 0/10 [00:00<?, ?it/s]


[GPU Check] Input tensors device: mps:0

[GPU Check] Model device: mps:0

[Sanity Check] Input: <|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Answer the following question: Find
\[\sin \left( \sin^{-1} \frac{3}{5} + \tan^{-1} 2 \right).\]<|im_end|>
<|im_start|>assistant


[Sanity Check] Output: To find \(\sin \left( \sin^{-1} \frac{3}{5} + \tan^{-1} 2 \right)\), we can use the angle addition formula for sine. The angle addition formula for sine states:

\[
\sin(a + b) = (\sin a)(\cos b) + (\cos a)(\sin b)
\]

Here, let \(a = \sin^{-1} \frac{3}{5}\) and \(b = \tan^{-1} 2\).

First, we need to find the values of \(\sin a\), \(\cos a\), and \(\cos b\).

Since \(a = \sin^{-1} \frac{3}5\), it implies that:
\[
\sin a = \frac{3}{5}
\]

To find \(\cos a\), we use the Pythagorean identity:
\[
\sin^2 a + \cos^2 a = 1
\]
\[
\left(\frac{3}{5}\right)^2 + \cos^2 a = 1
\] 
\[
\frac{9}{25} + \cos^2 a = 1
\]
\[
\cos^2 a = 1 - \frac

Problem 1580: 100%|██████████| 10/10 [07:35<00:00, 45.50s/it]



Problem 2/5


Problem 33: 100%|██████████| 10/10 [09:16<00:00, 55.66s/it]



Problem 3/5


Problem 2177:  40%|████      | 4/10 [18:51<28:17, 282.85s/it]


RuntimeError: probability tensor contains either `inf`, `nan` or element < 0

: 

In [ ]:
# Unload model and tokenizer from memory
del model
del tokenizer
gc.collect()

# Release GPU cache (if any)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [24]:
tokens = [tokenizer.decode(id) for id in output_ids[len(model_inputs.input_ids[0]):]]
entropies = reasoning_graph.metrics
visualize_tokens(tokens, entropies)

In [ ]:
# Example of loading a saved reasoning graph
loaded_graph = ReasoningGraph.load("/Users/adityashukzy/Documents/GitHub/MAT1510-Project/experiments/experiment_20251027_155017/problem_258/rollout_0")

# Verify the loaded data
print("Metrics length:", len(loaded_graph.metrics))
print("Probability distributions shape:", loaded_graph.prob_distributions[0].shape)
print("Node cutoff value:", loaded_graph.node_cutoff)

# You can now use this loaded graph for visualization or analysis

Metrics length: 185
Probability distributions shape: torch.Size([151936])
Node cutoff value: 0.0986328125


: 